In [ ]:
"""
Setup and data contract for surrogate selection analysis.

Goal: decide whether M(img_emb, src_emb, tar_emb, t*, t**) -> (PSNR, CLIP) can select a
per-image-best (t*, t**) cell, not just fit metrics globally. Selection only compares the
121 cells within one image, so downstream analysis uses the held-out TEST split (images not
seen during training). Loads the latest checkpoint, builds the 11x11 t-grid, precomputes
frozen embeddings, and batched grid predictions for every holdout image.
"""

import sys
from pathlib import Path

NOTEBOOK_DIR = Path.cwd()
if not (NOTEBOOK_DIR / "settings.py").exists():
    NOTEBOOK_DIR = NOTEBOOK_DIR / "models" / "modified-classification"
ROOT = NOTEBOOK_DIR.parents[1]
for p in (ROOT, NOTEBOOK_DIR):
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from scipy.stats import spearmanr

from model import MetricPredictor
from train import load_data, precompute_embeddings
from models.classification.settings import DEFAULT_T_START, DEFAULT_T_END
from settings import *

def fig_title(subtitle: str) -> str:
    return f"{subtitle} for {TARGET_METRIC_COL} on {METRICS_CSV.stem}, $\\delta={{{TARGET_T_DELTA}}}$"


# ---- latest run + sample-level holdout (full 11x11 grid per image) ----
# train.py splits rows 80/10/10, which leaves every image partially in train. Selection
# analysis needs complete grids and images not used for training so re-split by sample_id.
run_dirs = sorted(d for d in OUTPUTS_DIR.iterdir() if d.is_dir())
assert run_dirs, f"No run directories in {OUTPUTS_DIR}"
RUN_DIR = run_dirs[-1]
full_df = load_data()
all_sids = sorted(full_df.sample_id.unique())
rng = np.random.default_rng(SEED)
perm = rng.permutation(all_sids)
n_holdout = max(1, round(0.2 * len(perm)))
holdout_sids = sorted(perm[:n_holdout])
holdout_df = full_df[full_df.sample_id.isin(holdout_sids)].copy()
print(f"Run: {RUN_DIR.name}  holdout: {len(holdout_sids)} images  cells: {len(holdout_df)}")

# ---- t-grid ----
T_STAR = np.sort(holdout_df.t_start.unique())
T_STARSTAR = np.sort(holdout_df.t_end.unique())
N1, N2 = len(T_STAR), len(T_STARSTAR)
CELLS = N1 * N2
for sid, n in holdout_df.groupby("sample_id").size().items():
    assert n == CELLS, f"sample {sid} has {n} cells, expected {CELLS}"

SAMPLE_IDS = sorted(holdout_df.sample_id.unique())
N_IMG = len(SAMPLE_IDS)
SID_TO_K = {sid: k for k, sid in enumerate(SAMPLE_IDS)}
I_OF = {v: i for i, v in enumerate(T_STAR)}
J_OF = {v: j for j, v in enumerate(T_STARSTAR)}

DEFAULT_I = int(np.argmin(np.abs(T_STAR - DEFAULT_T_START)))
DEFAULT_J = int(np.argmin(np.abs(T_STARSTAR - DEFAULT_T_END)))
print(f"default cell index ({DEFAULT_I}, {DEFAULT_J}) -> t*= {T_STAR[DEFAULT_I]:.3g}, t**= {T_STARSTAR[DEFAULT_J]:.3g}")

# ---- load model ----
weights_path = RUN_DIR / "regressor_weights.pt"
ckpt = torch.load(weights_path, map_location="cpu", weights_only=False)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = MetricPredictor(freeze_encoders=FREEZE_ENCODERS, device=DEVICE)
model.regressor.load_state_dict(ckpt["regressor_state_dict"])
model.regressor.set_target_stats(ckpt["target_mean"], ckpt["target_std"])
model.regressor.to(DEVICE).eval()

emb = precompute_embeddings(holdout_df.drop_duplicates("sample_id"), model, DEVICE)

# ---- scalarization (standardize each axis before weighting) ----
W_PSNR, W_CLIP = 0.5, 0.5


def scalarize(psnr, clip, mu_sd=None):
    if mu_sd is None:
        zp = (psnr - psnr.mean()) / psnr.std()
        zc = (clip - clip.mean()) / clip.std()
    else:
        (pm, ps, cm, cs) = mu_sd
        zp = (psnr - pm) / ps
        zc = (clip - cm) / cs
    return W_PSNR * zp + W_CLIP * zc


def df_to_grid(df: pd.DataFrame, col: str) -> np.ndarray:
    out = np.full((N_IMG, N1, N2), np.nan)
    for row in df.itertuples():
        out[SID_TO_K[row.sample_id], I_OF[row.t_start], J_OF[row.t_end]] = getattr(row, col)
    return out


def predict_grid(img_emb: torch.Tensor, src_emb: torch.Tensor, tar_emb: torch.Tensor):
    """One batched forward pass; returns pred_psnr (N1,N2), pred_clip (N1,N2)."""
    tt1, tt2 = np.meshgrid(T_STAR, T_STARSTAR, indexing="ij")
    t1, t2 = tt1.reshape(-1), tt2.reshape(-1)
    n = CELLS
    img = img_emb.unsqueeze(0).expand(n, -1)
    src = src_emb.unsqueeze(0).expand(n, -1)
    tar = tar_emb.unsqueeze(0).expand(n, -1)
    t = torch.tensor(np.stack([t1, t2], axis=1), dtype=torch.float, device=DEVICE)
    with torch.no_grad():
        pred = model.regressor.denormalize(model.regressor(img, src, tar, t)).cpu().numpy()
    return pred[:, 0].reshape(N1, N2), pred[:, 1].reshape(N1, N2)


true_psnr = df_to_grid(holdout_df, "psnr")
true_clip = df_to_grid(holdout_df, "clip")
pred_psnr = np.zeros_like(true_psnr)
pred_clip = np.zeros_like(true_clip)
for sid in SAMPLE_IDS:
    k = SID_TO_K[sid]
    e = emb[sid]
    pp, pc = predict_grid(e["img"], e["src"], e["tar"])
    pred_psnr[k], pred_clip[k] = pp, pc

SCALAR_STATS = (true_psnr.mean(), true_psnr.std(), true_clip.mean(), true_clip.std())
true_m = scalarize(true_psnr, true_clip, SCALAR_STATS)
pred_m = scalarize(pred_psnr, pred_clip, SCALAR_STATS)
print(f"holdout grids: {true_psnr.shape}  ({N_IMG} images x {N1}x{N2} cells)")


In [ ]:
"""
Tier 1.1 — Per-image Spearman (distribution, not just mean).

The core competence: can M rank the 121 cells within each image? Global R² is a distraction
here. Spearman rho compares rank order of predicted vs true metrics on each image's grid.
A fat tail near or below zero means the model cannot rank cells for a chunk of images
regardless of aggregate fit.
"""

def per_image_spearman(true_grid, pred_grid):
    rhos = []
    for k in range(true_grid.shape[0]):
        rho, _ = spearmanr(true_grid[k].ravel(), pred_grid[k].ravel())
        rhos.append(np.nan if rho is None else float(rho))
    return np.array(rhos)

rho_psnr = per_image_spearman(true_psnr, pred_psnr)
rho_clip = per_image_spearman(true_clip, pred_clip)
rho_m = per_image_spearman(true_m, pred_m)

for name, r in [("PSNR", rho_psnr), ("CLIP", rho_clip), ("m", rho_m)]:
    rv = r[np.isfinite(r)]
    print(f"{name:5s}  median rho={np.nanmedian(r):.3f}  "
          f"frac<0.2={np.mean(rv < 0.2):.2f}  frac<0={np.mean(rv < 0):.2f}  n={len(rv)}")

fig, axes = plt.subplots(1, 3, figsize=(12, 3))
for ax, (name, r) in zip(axes, [("PSNR", rho_psnr), ("CLIP", rho_clip), ("m", rho_m)]):
    rv = r[np.isfinite(r)]
    if len(rv):
        ax.hist(rv, bins=min(30, max(3, len(rv))), edgecolor="white")
    else:
        ax.text(0.5, 0.5, "n/a", ha="center", va="center", transform=ax.transAxes)
    ax.axvline(0, color="k", lw=0.5)
    ax.set_title(f"per-image rho: {name}")
    ax.set_xlabel("Spearman rho")
fig.suptitle(fig_title("Per-image Spearman"), fontsize=14)
plt.tight_layout()
plt.show()


In [ ]:
"""
Tier 1.2 — Selection regret (the honest metric, in m units).

Regret is true m at the model-chosen cell minus true m at the real best cell, per image.
Lower is better; zero means the surrogate picked the optimal cell under the scalarized target.
"""

def regret(true_m_grid, pred_m_grid):
    out = np.zeros(true_m_grid.shape[0])
    for k in range(true_m_grid.shape[0]):
        chosen = np.unravel_index(pred_m_grid[k].argmax(), pred_m_grid[k].shape)
        best = true_m_grid[k].max()
        out[k] = best - true_m_grid[k][chosen]
    return out

reg = regret(true_m, pred_m)
print(f"regret  median={np.median(reg):.4f}  p90={np.percentile(reg, 90):.4f}  "
      f"frac~0(<1e-3)={np.mean(reg < 1e-3):.2f}")

fig, ax = plt.subplots(figsize=(6, 4))
ax.hist(reg, bins=min(30, max(5, N_IMG // 2)), edgecolor="white")
ax.set_xlabel("regret (m units)")
ax.set_ylabel("count")
ax.set_title("selection regret")
fig.suptitle(fig_title("Selection regret"), fontsize=14)
plt.tight_layout()
plt.show()


In [ ]:
"""
Tier 1.3 — Top-1 hit rate and top-3 overlap.

Measures whether the surrogate's favorite cell matches the true best (top-1) and whether the
true top-3 cells appear in the predicted top-3. These are stricter selection checks than
Spearman alone.
"""

def topk_metrics(true_m_grid, pred_m_grid, k=3):
    hit1, ov = [], []
    for i in range(true_m_grid.shape[0]):
        t_order = true_m_grid[i].ravel().argsort()[::-1]
        p_order = pred_m_grid[i].ravel().argsort()[::-1]
        hit1.append(t_order[0] == p_order[0])
        ov.append(len(set(t_order[:k]) & set(p_order[:k])) / k)
    return np.mean(hit1), np.mean(ov)

h1, ov3 = topk_metrics(true_m, pred_m, k=3)
print(f"top-1 hit rate={h1:.3f}   top-3 overlap={ov3:.3f}")


In [ ]:
"""
Tier 1.4 — Argmax-collapse test (the failure high R² hides).

If the predicted argmax clusters on one cell while the true argmax spreads across the grid,
the model learned a single global optimum and captured no per-image signal. Compare spread
of true vs predicted argmax locations; stop model work if predicted spread << true spread.
"""

def argmax_cells(grid):
    return np.array([np.unravel_index(grid[k].argmax(), grid[k].shape) for k in range(grid.shape[0])])

true_arg = argmax_cells(true_m)
pred_arg = argmax_cells(pred_m)

def spread(arg):
    return arg[:, 0].std() + arg[:, 1].std()

print(f"true argmax spread = {spread(true_arg):.3f}")
print(f"pred argmax spread = {spread(pred_arg):.3f}   (>> collapse if pred << true)")
print(f"unique pred cells = {len(set(map(tuple, pred_arg)))} / {CELLS}")

fig, axes = plt.subplots(1, 2, figsize=(8, 4), sharex=True, sharey=True)
for ax, arg, title in zip(axes, [true_arg, pred_arg], ["true argmax", "pred argmax"]):
    H = np.zeros((N1, N2))
    for i, j in arg:
        H[i, j] += 1
    ax.imshow(H, origin="lower", cmap="Blues")
    ax.set_title(title)
    ax.scatter([DEFAULT_J], [DEFAULT_I], c="r", marker="x", label="default")
    ax.set_xlabel("t_end idx")
    ax.set_ylabel("t_start idx")
    ax.legend(fontsize=8)
fig.suptitle(fig_title("Argmax collapse"), fontsize=14)
plt.tight_layout()
plt.show()


In [ ]:
"""
Tier 2.1 — Noise floor (label stability ceiling on regret).

Requires re-rendering a sample of cells at a second seed. You cannot rank better than the
labels are stable. If regret is on par with this noise floor, the bottleneck is data noise,
not architecture. Skipped here unless second-seed metrics are provided.
"""

SECOND_SEED_CSV = None  # e.g. Path("data/id_to_metrics_sdturbo_random10_seed2.csv")

if SECOND_SEED_CSV is None or not Path(SECOND_SEED_CSV).exists():
    print("SKIP: set SECOND_SEED_CSV to a re-rendered metrics file to compute the noise floor.")
    noise_floor_m = np.nan
else:
    second = pd.read_csv(SECOND_SEED_CSV)
    second = second.rename(columns={PSNR_COL: "psnr", CLIP_COL: "clip"})
    if TARGET_T_DELTA is not None:
        second = second[second.t_delta == TARGET_T_DELTA]
    # intersect holdout sample_ids
    second = second[second.sample_id.isin(SAMPLE_IDS)]
    s_psnr = df_to_grid(second, "psnr")
    s_clip = df_to_grid(second, "clip")
    delta_psnr = np.abs(true_psnr - s_psnr)
    delta_clip = np.abs(true_clip - s_clip)
    print(f"PSNR seed noise: median |Δ|={np.nanmedian(delta_psnr):.3f}")
    print(f"CLIP seed noise: median |Δ|={np.nanmedian(delta_clip):.4f}")
    noise_floor_m = np.nanmedian(np.abs(scalarize(true_psnr, true_clip, SCALAR_STATS)
                                        - scalarize(s_psnr, s_clip, SCALAR_STATS)))
    print(f"noise floor (m units) ~ {noise_floor_m:.4f}  vs regret median {np.median(reg):.4f}")


In [ ]:
"""
Tier 2.2 — CLIP resolution check (can the surrogate even see the CLIP axis?).

If the within-image CLIP range is not comfortably larger than CLIP MAE, CLIP-based selection
is guesswork. Same check for PSNR for completeness.
"""

clip_range = true_clip.max(axis=(1, 2)) - true_clip.min(axis=(1, 2))
psnr_range = true_psnr.max(axis=(1, 2)) - true_psnr.min(axis=(1, 2))
clip_mae = np.abs(pred_clip - true_clip).mean()
psnr_mae = np.abs(pred_psnr - true_psnr).mean()

print(f"within-image CLIP range: median={np.median(clip_range):.4f}  "
      f"p10={np.percentile(clip_range, 10):.4f}   (MAE={clip_mae:.3f})")
print(f"within-image PSNR range: median={np.median(psnr_range):.3f}  "
      f"p10={np.percentile(psnr_range, 10):.3f}   (MAE={psnr_mae:.3f})")

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].hist(clip_range, bins=min(30, max(5, N_IMG // 2)), edgecolor="white")
axes[0].axvline(clip_mae, color="r", linestyle="--", label=f"CLIP MAE={clip_mae:.3f}")
axes[0].legend()
axes[0].set_title("within-image CLIP range vs error")
axes[1].hist(psnr_range, bins=min(30, max(5, N_IMG // 2)), edgecolor="white")
axes[1].axvline(psnr_mae, color="r", linestyle="--", label=f"PSNR MAE={psnr_mae:.3f}")
axes[1].legend()
axes[1].set_title("within-image PSNR range vs error")
fig.suptitle(fig_title("Label resolution vs surrogate error"), fontsize=14)
plt.tight_layout()
plt.show()


In [ ]:
"""
Tier 2.3 — Target validity (does m agree with your eyes?).

For images where you visually identified a better edit, where does that cell rank under m?
If it is not near the top, fix scalarization (re-weight W_PSNR/W_CLIP) before trusting any
selection metric. Populate eyeball_cells with {image_index: (i, j)} grid indices.
"""

# image_index k in 0..N_IMG-1; (i, j) are grid indices into T_STAR / T_STARSTAR
eyeball_cells: dict[int, tuple[int, int]] = {
    # 0: (DEFAULT_I, DEFAULT_J),
}

if not eyeball_cells:
    print("SKIP: populate eyeball_cells with human-preferred (i, j) grid indices.")
else:
    ranks = []
    for k, (i, j) in eyeball_cells.items():
        order = true_m[k].ravel().argsort()[::-1]
        flat_idx = np.ravel_multi_index((i, j), (N1, N2))
        rank = int(np.where(order == flat_idx)[0][0])
        ranks.append(rank)
    ranks = np.array(ranks)
    print(f"human-preferred cell rank under m: median={np.median(ranks):.0f}/{CELLS}  "
          f"frac in top-5={np.mean(ranks < 5):.2f}")


In [ ]:
"""
Tier 3.1 — Fresh-seed realized gain (the certifying number).

Take M's chosen t on held-out images, render with ChordEdit at a seed not used in training,
score, and compare to the default cell. If selecting via M does not beat default here,
within-image ranking is not yet actionable. Requires a render_and_score hook — skipped until
wired to the pipeline.
"""

FRESH_SEED = None  # set e.g. 12345

def render_and_score(image_k: int, t_star: float, t_starstar: float, seed: int):
    raise NotImplementedError("Wire to ChordEditPipeline.render + metric scoring")

if FRESH_SEED is None:
    print("SKIP: set FRESH_SEED and implement render_and_score() to compute realized gain.")
else:
    realized = []
    for k in range(N_IMG):
        i, j = np.unravel_index(pred_m[k].argmax(), pred_m[k].shape)
        psnr_c, clip_c = render_and_score(k, T_STAR[i], T_STARSTAR[j], seed=FRESH_SEED)
        psnr_d, clip_d = render_and_score(k, T_STAR[DEFAULT_I], T_STARSTAR[DEFAULT_J], seed=FRESH_SEED)
        realized.append(
            scalarize(np.array([psnr_c]), np.array([clip_c]), SCALAR_STATS)[0]
            - scalarize(np.array([psnr_d]), np.array([clip_d]), SCALAR_STATS)[0]
        )
    realized = np.array(realized)
    nf = noise_floor_m if not np.isnan(noise_floor_m) else 0.0
    print(f"realized Δm vs default: median={np.median(realized):.4f}  "
          f"frac>0={np.mean(realized > 0):.2f}  frac>noise_floor={np.mean(realized > nf):.2f}")
    fig, ax = plt.subplots(figsize=(6, 4))
    ax.hist(realized, bins=min(30, max(5, N_IMG // 2)), edgecolor="white")
    ax.axvline(0, color="k")
    ax.set_title("realized gain over default")
    fig.suptitle(fig_title("Fresh-seed realized gain"), fontsize=14)
    plt.tight_layout()
    plt.show()


In [ ]:
"""
Tier 3.2 — Deviate-or-default gating (precision/recall).

High precision matters: do not degrade easy images chasing phantom gains. Flags images where
predicted gain over default exceeds the noise floor; compare against images truly improvable
under ground-truth m.
"""

nf = noise_floor_m if "noise_floor_m" in dir() and not np.isnan(noise_floor_m) else 0.0
default_m = true_m[:, DEFAULT_I, DEFAULT_J]
truly_improvable = (true_m.max(axis=(1, 2)) - default_m) > nf
pred_gain = pred_m.max(axis=(1, 2)) - pred_m[:, DEFAULT_I, DEFAULT_J]
flagged = pred_gain > nf

tp = int(np.sum(flagged & truly_improvable))
precision = tp / max(int(flagged.sum()), 1)
recall = tp / max(int(truly_improvable.sum()), 1)
print(f"deviate gate  precision={precision:.3f}  recall={recall:.3f}  "
      f"({flagged.sum()} flagged of {N_IMG})")
print(f"noise floor used: {nf:.4f}")


In [ ]:
"""
Tier 4.1 — Calibration per head (only if Tiers 1–3 pass).

Bin predicted values and compare mean predicted vs mean true in each bin. Compression toward
the mean (flat line) is the signature of argmax collapse.
"""

fig, axes = plt.subplots(1, 2, figsize=(8, 4))
for ax, true_g, pred_g, name in [
    (axes[0], true_psnr, pred_psnr, "PSNR"),
    (axes[1], true_clip, pred_clip, "CLIP"),
]:
    p, t = pred_g.ravel(), true_g.ravel()
    bins = np.quantile(p, np.linspace(0, 1, 11))
    bins[0] -= 1e-6
    bins[-1] += 1e-6
    idx = np.digitize(p, bins)
    bx, by = [], []
    for b in range(1, 11):
        mask = idx == b
        if mask.any():
            bx.append(p[mask].mean())
            by.append(t[mask].mean())
    ax.plot(bx, by, "o-")
    lo, hi = min(bx + by), max(bx + by)
    ax.plot([lo, hi], [lo, hi], "k--", alpha=0.4)
    ax.set_title(f"calibration: {name}")
    ax.set_xlabel("pred")
    ax.set_ylabel("true")
fig.suptitle(fig_title("Per-head calibration"), fontsize=14)
plt.tight_layout()
plt.show()


In [ ]:
"""
Tier 4.2 — Surface-shape fidelity (pred vs true heatmaps).

Random holdout images: compare true and predicted PSNR/CLIP surfaces. You are checking shape
(monotone trends, trade-off ridge), not global scale. A flat predicted surface indicates
collapse.
"""

rng = np.random.default_rng(SEED)
pick = rng.choice(N_IMG, size=min(4, N_IMG), replace=False)
for k in pick:
    fig, axes = plt.subplots(2, 2, figsize=(7, 6))
    panels = [
        (axes[0, 0], true_psnr[k], "true PSNR"),
        (axes[0, 1], pred_psnr[k], "pred PSNR"),
        (axes[1, 0], true_clip[k], "true CLIP"),
        (axes[1, 1], pred_clip[k], "pred CLIP"),
    ]
    for ax, g, title in panels:
        im = ax.imshow(g, origin="lower", cmap="viridis")
        ax.set_title(f"img {SAMPLE_IDS[k]}: {title}")
        fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    fig.suptitle(fig_title(f"Surface fidelity (sample {SAMPLE_IDS[k]})"), fontsize=13)
    plt.tight_layout()
    plt.show()


In [ ]:
"""
Tier 4.3 — Error stratification (regret vs default quality and grid-edge optima).

Diagnose whether failures cluster on low-default-quality images or when the true best cell
lies on the grid boundary (suggesting the grid may be too narrow). edit_type grouping requires
external labels — omitted until annotated.
"""

df_err = pd.DataFrame({
    "sample_id": SAMPLE_IDS,
    "regret": reg,
    "default_quality": true_m[:, DEFAULT_I, DEFAULT_J],
    "best_at_edge": [
        (i in (0, N1 - 1)) or (j in (0, N2 - 1)) for i, j in true_arg
    ],
})

print("regret vs default quality (corr):",
      np.corrcoef(df_err["default_quality"], df_err["regret"])[0, 1])
print("regret when true best is at grid edge:",
      df_err.groupby("best_at_edge")["regret"].median().to_dict())
print(df_err.sort_values("regret", ascending=False).head())

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].scatter(df_err["default_quality"], df_err["regret"], alpha=0.7)
axes[0].set_xlabel("default m")
axes[0].set_ylabel("regret")
axes[0].set_title("regret vs default quality")
edge_labels = ["interior", "edge"]
edge_vals = [df_err.loc[~df_err.best_at_edge, "regret"],
             df_err.loc[df_err.best_at_edge, "regret"]]
axes[1].boxplot(edge_vals, tick_labels=edge_labels)
axes[1].set_ylabel("regret")
axes[1].set_title("regret by true-argmax location")
fig.suptitle(fig_title("Error stratification"), fontsize=14)
plt.tight_layout()
plt.show()


In [ ]:
"""
Decision summary — run tiers in order.

Tier 1 (1.1 Spearman / 1.2 regret / 1.4 collapse) decides whether M can rank cells within
each image. Tier 2 (noise floor, target validity) bounds what is achievable. Tier 3.1
(fresh-seed gain) certifies the selector beats default in practice. Tier 4 diagnoses how
failures manifest once you know whether selection works.
"""

print(fig_title("Decision summary"))
print(f"  holdout images: {N_IMG}   grid: {N1}x{N2}={CELLS}")
print(f"  Tier 1.1 median rho (m): {np.median(rho_m):.3f}")
print(f"  Tier 1.2 regret median:   {np.median(reg):.4f}")
print(f"  Tier 1.3 top-1 / top-3:   {h1:.3f} / {ov3:.3f}")
print(f"  Tier 1.4 spread true/pred: {spread(true_arg):.3f} / {spread(pred_arg):.3f}")
print(f"  Tier 3.2 precision/recall: {precision:.3f} / {recall:.3f}")
